# 6.3 Imaging And Inversion

This tutorial introduces `frequensolve.imaging`: the API that declares a seismic inverse problem once and then works with short scalar calls and SciPy-style linear operators. It builds a small acoustic acquisition with a true and a starting model, defines a control space over the sediment velocity profile, generates observed frequency-domain data, linearizes the misfit, applies the Jacobian and its adjoint, images a sensitivity kernel on a Cartesian grid, and runs a two-stage full waveform inversion (FWI) with checkpoints.

The purpose is practical: after this notebook, you should know which Python object owns each inversion decision, how observed and simulated traces are paired, what a linearization is and what hangs off it, and which artifacts to inspect before scaling to a production inversion.

## How To Read This Tutorial

Imaging is where FrequenSolve stops being a forward-modeling wrapper and becomes an inversion platform. The project/simulation/job structure is unchanged, but one new object, the `ImagingProblem`, binds the simulation, the control space, the observed data, the misfit, the frequencies and the execution site. Everything else derives from it: values, gradients, Jacobian and normal operators, and the FWI, LSRTM, RTM and sensitivity-kernel workflows.

Read this notebook in two layers: first as an authoring tutorial (models, control blocks, observed data, misfit, all inspectable without a solver), then as a strict solver workflow that generates observed data, linearizes, images and inverts.

## Imaging Vocabulary

| Concept | Python API | What it controls |
| --- | --- | --- |
| Starting simulation | `project.new_simulation(...)` | The model, mesh, acquisition and wavelet that the problem linearizes around. |
| Control block | `im.DepthProfile(...)`, `im.GridParameters(...)`, `im.InterfaceParameters(...)`, `im.SourceParameters(...)` | One Sauce control block: where it lives (a subdomain or surface), its basis (`spacing`, `count` or `nodes`), and optionally a `transform` and physical `limits`. |
| Control space | `im.ControlSpace(vp=..., rho=...)` | The ordered collection of blocks that defines the real optimizer vector. |
| State versus vector | `im.ControlState`, `im.ControlVector` | The complete baseline over every block, versus a vector on the active blocks of a stage (gradients, directions, updates). |
| Observed data | `im.ObservedData(observed_job)` | Frequency-domain traces paired with the simulated receiver groups by name; frequencies are inferred from the job. |
| Misfit | `im.Misfit.huber(...)`, `im.Preprocess.offset_taper(...)` | Loss, comparison, normalization and preprocessing hooks, one to one with Sauce's objective terms. |
| Problem | `im.ImagingProblem(sim, controls=..., observed=..., misfit=..., frequencies=..., site=...)` | The single binding object every workflow derives from. |
| Linearization | `problem.linearize(v)` | One saved solver state per frequency: `value`, `gradient`, `jacobian`, `normal`. |
| Operators | `J @ dv`, `J.H @ r`, `H @ dv` | SciPy `LinearOperator`s over control and data spaces. |
| Workflows | `im.rtm(...)`, `im.sensitivity_kernel(...)`, `im.FWI(...)`, `im.LSRTM(...)` | The outer loops: single-linearization images and staged inversion with checkpoints. |
| Image reader | `im.ImageSet` | Cartesian-grid images as `xarray.Dataset` objects (`raw`, `smoothed`). |

## Data Pairing Is The Central Contract

An inverse problem is meaningful only if observed and simulated receiver groups are paired correctly. The observed data may come from field traces, a previously run synthetic job, or a fetched remote result; the simulated side comes from the starting simulation. The misfit pairs the two by receiver-group name.

Before a large run, inspect the exported misfit contract and the control payload: receiver-group names, observed paths, frequencies, hooks, and the active control blocks. Most inversion mistakes are contract mistakes before they are numerical mistakes.

## Imports And Project Path

The notebook writes a small scratch project under `./scratch/tutorials/imaging`. The imaging API is imported as `im`; the same names are also available at the package root (`fs.ImagingProblem`, `fs.DepthProfile`, ...).

In [ ]:
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display

import frequensolve as fs
from frequensolve import imaging as im

u = fs.ureg

In [ ]:
project = fs.Project(
    name="project",
    pretty_name="Imaging Tutorial",
    path="./scratch/tutorials/imaging",
    log_level="INFO",
    log_to_console=True,
)

project.path

## Build Matched True And Starting Simulations

An inversion tutorial needs two models:

- a **true** model, which stands in for field data or a higher-fidelity synthetic experiment and is only used to generate observed data;
- a **starting** model, which the problem linearizes around and the inversion updates.

Both simulations share the geometry, mesh, acquisition and boundary conditions: a 100 m water column over a 500 m sediment column. The sediment velocity is authored as a vertical profile (`xarray.DataArray` against depth) so that the true and starting models differ only in that profile. The true sediment is layered with a low-velocity notch; the start is a smooth linear ramp through the same end points. That mismatch is exactly what the inversion should recover.

In [ ]:
WATER_DEPTH = 0.1  # km
MODEL_DEPTH = 0.6  # km


def sediment_truth(below):
    # Layered sediment Vp (km/s) by depth below the seabed (km), with a low-velocity notch.
    below = np.asarray(below, dtype=float)
    vp = np.full_like(below, 2.4)
    vp[below < 0.35] = 2.2
    vp[below < 0.25] = 1.8  # the notch
    vp[below < 0.15] = 2.0
    vp[below < 0.05] = 1.7
    return vp


def sediment_start(below):
    # Smooth starting Vp: a linear ramp through the truth's end points.
    below = np.asarray(below, dtype=float)
    return 1.7 + (2.4 - 1.7) * below / (MODEL_DEPTH - WATER_DEPTH)


def build_simulation(project, *, name, sediment_vp):
    sim = project.new_simulation(
        name=name,
        physics="acoustic",
        dimension=2,
        units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
    )

    below = np.linspace(0.0, MODEL_DEPTH - WATER_DEPTH, 201)
    vp_column = xr.DataArray(
        sediment_vp(below), dims=["z"], coords={"z": WATER_DEPTH + below}
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.2])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(
        name="water",
        properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3},
    )
    model.add_surface(name="seabed", depth=WATER_DEPTH * u.km)
    model.add_layer(
        name="sediment",
        properties={"Vp": vp_column, "Rho": 2.0 * u.g / u.cm**3},
    )
    model.add_surface(name="bottom", depth=MODEL_DEPTH * u.km)
    sim += model

    sim += model.hex_mesh_generator([12, 6])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=3.0, f_high=10.0)

    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(
        conditions=["pml"],
        boundaries=["x_min", "x_max", "z_max"],
        pml_wavelengths=0.75,
    )

    acq = fs.Acquisition()
    acq.add_sources(
        kind="scalar",
        coords=fs.Q_([[0.2, 0.02], [0.6, 0.02], [1.0, 0.02]], "km"),
    )
    hydrophone = fs.ReceiverNode(name="hydrophone")
    hydrophone.add_component(name="p", field="pressure")
    receiver_coords = [fs.Q_([x, 0.05], "km") for x in np.linspace(0.1, 1.1, 51)]
    acq.add_receiver_group(name="surface", device=hydrophone, coords=receiver_coords)
    sim += acq

    sim += fs.SolverConfig(tolerance=1.0e-4)
    return sim


true_sim = build_simulation(project, name="imaging_true", sediment_vp=sediment_truth)
start_sim = build_simulation(project, name="imaging_start", sediment_vp=sediment_start)

## Inspect The True And Starting Models

The starting model is what the problem linearizes around. The true model is only used here to synthesize observed data; in a field-data workflow it is replaced by a path to preprocessed frequency-domain traces whose receiver-group names match the starting simulation's acquisition.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), constrained_layout=True)
true_sim.model.plot("vp", ax=axes[0], aspect="equal")
axes[0].set_title("True model: Vp")
start_sim.model.plot("vp", ax=axes[1], aspect="equal")
axes[1].set_title("Starting model: Vp")

below = np.linspace(0.0, MODEL_DEPTH - WATER_DEPTH, 201)
axes[2].plot(sediment_truth(below), below, label="true")
axes[2].plot(sediment_start(below), below, "--", label="start")
axes[2].invert_yaxis()
axes[2].set_xlabel("Vp [km/s]")
axes[2].set_ylabel("depth below seabed [km]")
axes[2].legend()
axes[2].set_title("Sediment profile");

## Frequencies

Frequency-domain inversion works on a selected set of frequencies. Three low frequencies keep the tutorial fast and the contracts readable; the FWI section later uses them as two continuation bands. The same list is used for the observed-data job and for the problem, so that every observed frequency is available to every stage.

In [ ]:
frequencies = [4.0, 6.0, 8.0]

## Author The Control Space

A control block names one Sauce control-registry block: where it lives (a subdomain or surface), its basis (exactly one of `spacing`, `count`, `nodes`) and, optionally, a `transform` and physical `limits`. Extent is never authored; it is the subdomain's span. Here one `DepthProfile` puts a hat-function profile on the sediment velocity every 50 m below the seabed, with a `log` transform so that zero coefficients reproduce the starting model exactly and updates stay positive.

Binding the space to the starting simulation resolves the blocks and shows the `controls` payload the solver receives. The problem does this binding itself; it is shown here so the contract can be reviewed before any job is submitted.

In [ ]:
controls = im.ControlSpace(
    vp=im.DepthProfile("vp", "sediment", spacing=0.05 * u.km, transform="log"),
)

bound = controls.bind(start_sim)
{"keys": controls.keys, "blocks": bound.qualified_names, "payload": bound.controls_payload()}

## Observed Data And Misfit

Observed data is declared from the job that will produce it. Passing a trace-producing `FrequencyDomainJob` lets FrequenSolve infer both the observed trace path and the frequency list; the job only needs to have run by the time the problem is linearized.

The misfit maps one to one onto Sauce's objective terms. This tutorial uses a Huber loss (robust to a few outlying traces) with a near-offset taper so that the direct arrival does not dominate the residual. The resolved contract below is what the solver's `Imaging.misfit` block will contain; check the receiver-group names and observed paths here.

In [ ]:
observed_job = fs.FrequencyDomainJob(
    name="observed_true",
    simulation=true_sim,
    f_list=frequencies,
)
observed = im.ObservedData(observed_job)

misfit = im.Misfit.huber(
    delta=1.345,
    preprocess=[im.Preprocess.offset_taper(d0=0.1 * u.km, d1=0.25 * u.km)],
)

print("observed frequencies:", observed.frequencies)
pprint(misfit.to_fs(observed.resolve(start_sim)))

## Strict Solver Run: Generate Observed Data

The remaining cells are the real solver path. They are intentionally strict: if the configured site's solver or imaging workflow is unavailable, the notebook should fail here and leave logs, job JSON and result paths for inspection.

First run the frequency-domain job on the true model. Everything the imaging API submits afterwards goes through the same `site`.

In [ ]:
site = fs.Site()

observed_result = site.submit(observed_job).wait()
observed_traces = observed_result.traces()
observed_traces.summary

## Declare The Problem

`im.ImagingProblem` binds the starting simulation, the control space, the observed data, the misfit, the frequencies and the site once. It deep-copies the simulation, resolves the blocks and fixes the vector layout. Nothing is submitted yet: `problem.space` is the resolved control space, `problem.state` the complete baseline (zero coefficients, since the start model is the reference), and `problem.capabilities()` a static check of the block/misfit/physics combination that reports what Sauce would reject.

In [ ]:
problem = im.ImagingProblem(
    start_sim,
    controls=controls,
    observed=observed,
    misfit=misfit,
    frequencies=frequencies,
    site=site,
    workdir=Path(project.path) / "fwi",
    name="fwi_tutorial",
)

print("blocks:", problem.space.blocks, "size:", problem.space.size)
print("capabilities:", problem.capabilities())
display(problem.state.to_xarray("vp"))

## Linearize: Value, Gradient, And The RTM Image

`problem.linearize(v)` saves one solver state per frequency at the point `v` (the current state when omitted) and returns a `Linearization`. Its `value` is the misfit, `report` splits it by objective term, and `gradient` is the real covector on the active blocks. Linearizations are cached by fingerprint, so `problem.value()`, `problem.gradient()` and the operators below at the same point cost one job family.

The gradient is Sauce's covector in the `simulated - observed` convention, which is exactly what `im.rtm(problem)` returns; its negative is the classic RTM image on the control space. On a depth profile the "image" is a 1-D sensitivity curve: it should light up where the starting and true sediments differ.

In [ ]:
lin = problem.linearize()
gradient = lin.gradient

print("misfit value:", lin.value)
print("per term:", lin.report)

fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)
(-gradient).plot("vp", ax=ax)
ax.set_title("RTM image on the Vp profile (-gradient)")
display(gradient.to_xarray("vp"))

In [ ]:
image = im.rtm(problem)  # the same covector, reused from the cache
np.allclose(np.asarray(image), np.asarray(gradient))

## Jacobian And Normal Operators

`lin.jacobian` is a SciPy `LinearOperator` between the control space and the data space: `J @ dv` (a Jacobian-vector product, Sauce's `jvp`) returns a complex `DataVector`; `J.H @ r` (a vector-Jacobian product, `vjp`) returns the real covector. `lin.normal` is the frozen Gauss-Newton operator `Re(J^H W J)`, self-adjoint on the control space.

The cell applies both to a random direction and checks the Gauss-Newton identity `J.H @ (J @ dv) == H @ dv` (up to the misfit weighting, which is unit here). Operators compose with `@`, `+` and scalars, so `H + alpha * R.T @ R` drops straight into `scipy.sparse.linalg.cg`.

In [ ]:
J = lin.jacobian
H = lin.normal

dv = problem.space.random(seed=1)
d_lin = J @ dv          # DataVector over (frequency, source, component, receiver)
g_gn = J.H @ d_lin      # ControlVector
h_dv = H @ dv

print("J @ dv:", d_lin.shape, "as dataset:")
display(d_lin.to_dataset())
print("max |J.H (J dv) - H dv|:", float(np.max(np.abs(np.asarray(g_gn) - np.asarray(h_dv)))))

## Sensitivity Kernel On A Cartesian Grid

`im.sensitivity_kernel` images the model sensitivity on a regular grid that is independent of the finite-element mesh and of the control space. With the default `observed=None` the solver uses zero observed data, so the image is the pure sensitivity kernel of the data with respect to `vp`; `observed=True` images the misfit residual instead. The result is an `ImageSet` whose `raw` dataset holds one `(z, x)` array per requested property.

In [ ]:
grid = fs.CartesianGrid(n=[121, 61], x0=[0.0, 0.0], x1=[1.2, 0.6])

kernels = im.sensitivity_kernel(
    problem, grid, properties=["vp"], condition="fwi", frequencies=[6.0]
)
raw = kernels.raw

fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
raw["vp"].plot.imshow(ax=ax, x="x", y="z", yincrease=False, cmap="RdBu_r")
ax.set_title("vp sensitivity kernel at 6 Hz")
ax.set_xlabel("x [km]")
ax.set_ylabel("z [km]")
display(raw)

## Two-Stage FWI With Checkpoints

A `Stage` names the frequencies, the iteration budget and the active blocks of one continuation stage. `Stage.bands` builds one stage per frequency band: the first inverts the 4 Hz data alone, the second adds 6 Hz. `im.FWI` runs the stages with projected L-BFGS (an RMS step cap per block keeps the log updates small), a first-order Tikhonov penalty on the profile, and a checkpoint that is written after every accepted iteration.

`run(resume=True)` picks up a checkpoint written by an earlier run of the same problem and stage list: completed stages are skipped and an interrupted stage continues with its remaining budget. Rerunning this cell after an interruption therefore continues rather than restarts.

In [ ]:
stages = im.Stage.bands([[4.0], [4.0, 6.0]], iterations=[3, 3], active=["vp"])

fwi = im.FWI(
    problem,
    stages=stages,
    optimizer=im.LBFGS(memory=5, step_limit=0.05),
    penalty=im.Tikhonov(alpha=1.0e-2, order=1),
    checkpoint="checkpoint.h5",
    history="history.json",
)
result = fwi.run(resume=True)

for stage in result.stages:
    print(
        f"{stage.name}: frequencies={stage.frequencies} iterations={stage.iterations} "
        f"linearizations={stage.linearizations} data loss {stage.initial_loss.data:.4g} -> "
        f"{stage.final_loss.data:.4g} ({stage.message})"
    )
print("success:", result.success, "checkpoint:", result.checkpoint)

## Inspect The Recovered Profile

`result.state` is the complete control state and `result.vector()` its active slice. On a log-transformed profile the recovered velocity at the control nodes is the starting profile times `exp(coefficients)`; `result.simulation` is the starting simulation with those coefficients installed, ready for further jobs. Compare the recovered profile with the truth and the start: with three low frequencies and a handful of iterations the notch should begin to appear, not be resolved.

In [ ]:
final = result.vector()
nodes = final.to_xarray("vp")
node_depths = np.asarray(nodes.coords[nodes.dims[0]])
recovered = sediment_start(node_depths) * np.exp(np.asarray(nodes.values))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
below = np.linspace(0.0, MODEL_DEPTH - WATER_DEPTH, 201)
axes[0].plot(sediment_truth(below), below, label="true")
axes[0].plot(sediment_start(below), below, "--", label="start")
axes[0].plot(recovered, node_depths, "o-", label="recovered")
axes[0].invert_yaxis()
axes[0].set_xlabel("Vp [km/s]")
axes[0].set_ylabel("depth below seabed [km]")
axes[0].legend()
axes[0].set_title("Sediment profile")
result.simulation.model.plot("vp", ax=axes[1], aspect="equal")
axes[1].set_title("Recovered model: Vp")

display(nodes)

## Before Moving On

Before scaling an inversion, inspect the contracts more carefully than the final picture. The active control blocks and their support, the observed/simulated receiver-group pairing, the misfit hooks, the frequencies of each stage and the per-stage loss reduction determine whether a result is meaningful.

The problem caches every linearization by fingerprint under its `workdir`; the checkpoint and history files live there too, so an interrupted run can be resumed from a fresh session by declaring the same problem and stages again.

## Review Checklist

Before scaling this workflow, inspect these artifacts:

| Artifact | What to check |
| --- | --- |
| `controls.bind(start_sim).controls_payload()` | The active block names and their order; one block per property and subdomain you intend to invert. |
| `misfit.to_fs(observed.resolve(start_sim))` | Receiver-group names, observed paths, loss, normalization and preprocessing hooks. |
| `problem.capabilities()` | No errors; warnings name documented limitations of the block/misfit/physics combination. |
| `problem.state.to_xarray("vp")`, `problem.space.support` | Node coordinates of the profile and the support mask: frozen nodes drop out of the optimizer vector. |
| `lin.value`, `lin.report`, `lin.gradient` | The misfit and its split per term; the gradient lights up where the models differ. |
| `J.H @ (J @ dv)` versus `H @ dv` | The Jacobian, its adjoint and the normal operator are consistent. |
| `kernels.raw` | Image names and `(z, x)` dimensions on the requested grid. |
| `result.stages`, `history.json` | Per-stage loss reduction, iteration counts and termination messages. |
| `checkpoint.h5` | The run can be resumed; a checkpoint of a different problem or stage list is rejected. |
| Logs and result directory | Solver-side failures are debugged from the strict run artifacts, not hidden in the notebook. |